In [ ]:
import kagglehub
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
numerical_cols = df.select_dtypes(include=["int64","float64"]).columns.drop("Target")
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].mean())

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
# no need to encoding

In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Target'].dropna(), bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, shuffle=True,)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

In [ ]:
model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred, zero_division=0)

In [ ]:
print(f"  F1-Score:  {f1:.4f}")

In [ ]:
# Task 1: Write your code here:
feature_cols=df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(20, 10))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
model.feature_importances_>10 # note that the first is only true

In [ ]:
feature_cols # here the first is

In [ ]:
# so the Golden Feature is ( P_2 )

In [ ]:
Golden_Feature= feature_importance.filter(model.feature_importances_>10)
Golden_Feature

In [ ]:
# Task Bonus: Write your code here: